In [ ]:
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests

from _load_brain_atlases import load_brain_atlases
from _results_computation import (read_pickle, compare_conditions, threshold_schaefers,
                                  save_schaefer_maps, plot_shared_voxels)

In [ ]:
# load in records
# comparing (recursion - iteration) and (prose - list)
records = read_pickle("output/organized_results_by_condition.pkl")
nulls = read_pickle('output/null_distribution_condition_intersections.pkl')
atlases = load_brain_atlases()

In [ ]:
# groups
groups = {
    # recursion similar in prose and code:
    # P101, P102, P104, P107, P108, P110, P113, P116, P121, P122, P124, P125
    'similar'      : ['102', '104', '107', '108', '110', '113', '116', '121', '122', '124', '125'],
    # recursion different
    # P101, P105, P109, P112, P113, P114, P115, P118, P120, P124, P125, P127, P130, P131, P203
    'different'    : ['105', '109', '112', '113', '114', '115', '118', '120', '124', '125', '127', '130', '131', '203'],
    # visualization
    # P102, P107, P108, P115, P118, P120, P121, P124, P127, P130, P203
    'visualization': ['102', '107', '108', '115', '118', '120', '121', '124', '127', '130', '203'],
    # planning
    # P107, P110, P112, P113, P114, P116, P122, P125, P133
    'planning'     : ['107', '110', '112', '113', '114', '116', '122', '125', '133'],
}

# 102, 106, 109, and 116 aren't significantly different from chance
not_significant = ['102', '106', '109', '116']

In [ ]:
def normalize_by_group_sizes(schaefers, group):
    # normalize since the group sizes are different
    norm_schaefers = {}
    for condition,counts in schaefers.items():
        norm_counts = np.array([n/len(group) for n in counts])
        norm_schaefers[condition] = norm_counts
    return norm_schaefers

def analyze_group(group_name, alpha=0.05):
    group = groups[group_name]
    results = compare_conditions(records, nulls, 'recursion', 'prose',
                                 control_a='iteration', control_b='list',
                                 participants=group, exclude=not_significant)

    # correct for multiple comparisons
    results['reject'], results['qvals'], _, _ = multipletests(
        np.nan_to_num(results['pvals'], nan=1.0), alpha=0.001, method='fdr_bh')

    # Group-level significance for the schaefer parcel tallies
    # (see threshold_schaefers in _results_computation.py)
    results['schaefers_thresholded'], results['schaefer_qvals'] = threshold_schaefers(
        results['schaefers'], nulls, results['participant_ids'], results['key_by_subset'], alpha=alpha)

    results['normalized'] = normalize_by_group_sizes(results['schaefers'], group)
    results['normalized_thresholded'] = normalize_by_group_sizes(results['schaefers_thresholded'], group)
    return results

In [ ]:
group_results = {name: analyze_group(name) for name in ['similar', 'different']}

for name, results in group_results.items():
    print(name)
    for pid, pval in zip(results['participant_ids'], results['pvals']):
        print(pid, pval)
    print(results['reject'])
    for subset, qvals in results['schaefer_qvals'].items():
        sig_parcels = np.where(qvals < 0.05)[0]
        print(f"{subset}: {len(sig_parcels)} parcels survive q<0.05 -> {sorted(sig_parcels.tolist())}")
    print()

In [ ]:
for name, results in group_results.items():
    plot_shared_voxels(results['participant_ids'], results['observed_vals'], results['null_vals_list'],
                       title=f'Top-modeled Voxels Shared Between Recursion and Prose ({name})')

In [ ]:
# counts are normalized by group size, so the colormap runs 0-1
save_schaefer_maps(group_results['similar']['normalized'], atlases, vmax=1)

In [ ]:
save_schaefer_maps(group_results['different']['normalized'], atlases, vmax=1)

In [ ]:
save_schaefer_maps(group_results['similar']['normalized_thresholded'], atlases, vmax=1)

In [ ]:
save_schaefer_maps(group_results['different']['normalized_thresholded'], atlases, vmax=1)

In [ ]:
def decide_t_test(g1_data, g2_data):  # based on variance, decide which t-test to use
    g1var = np.nanvar(g1_data, ddof=1)
    g2var = np.nanvar(g2_data, ddof=1)
    larger = max(g1var, g2var)
    smaller = min(g1var, g2var)
    ratio = larger/smaller
    test = "Student's" # only using Welch's so t-values can be compared
    equal_var = True
    if ratio >= 4:
        test = "Welch's"
        equal_var = False
    return test, equal_var

# participants who are in both groups (113, 124, 125) are only counted in 'similar'
similar_res, different_res = group_results['similar'], group_results['different']
intersects = {
    'similar'  : similar_res['observed_vals'],
    'different': [n for pid, n in zip(different_res['participant_ids'], different_res['observed_vals'])
                  if pid not in groups['similar']],
}

test,equal_var = decide_t_test(intersects['similar'], intersects['different'])

t_stat,p_value = stats.ttest_ind(intersects['similar'], intersects['different'], equal_var=equal_var)
print(f"Similar mean: {np.mean(intersects['similar']):.3f}, Different mean: {np.mean(intersects['different']):.3f} ($t={t_stat:.3f}$, $p={p_value:.3f}$)")